# Phase 5 on a free Kaggle GPU: fine-tune the student on the train labels

Trains a LoRA adapter with `train/train_lora.py` on the teacher's labels for the 20k train rows,
then scores it on the 200-row gold set with `train/predict.py`.

**Before running**
1. Settings: *Accelerator* = **GPU T4** (one is enough), *Internet* = **on**.
2. Add as inputs: the dataset with `train.parquet` / `val.parquet`, and a dataset with the
   teacher labels `train.jsonl` (prompt v2) from `label_kaggle.ipynb`.
3. Download `models/` and `predictions/` from the output when it finishes.

In [ ]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # or Qwen/Qwen2.5-1.5B-Instruct (~3x slower)
EPOCHS = 1
LR = 2e-4
REPO = "https://github.com/bhzcode1/customer-s-complaint.git"

In [ ]:
!git clone -q {REPO} /kaggle/working/repo
!pip install -q "peft>=0.17" duckdb datasketch

In [ ]:
import os, shutil, glob
os.chdir("/kaggle/working/repo")
for d in ("data/processed", "data/labels"):
    os.makedirs(d, exist_ok=True)
for name, dest in (("train.parquet", "data/processed"), ("val.parquet", "data/processed"),
                   ("train.jsonl", "data/labels")):
    found = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not found:
        raise FileNotFoundError(f"{name} not found under /kaggle/input (Add Input)")
    shutil.copy(found[0], dest)
    print(f"{name} <- {found[0]}")

In [ ]:
!python train/train_lora.py --model {MODEL} --labels data/labels/train.jsonl --split train     --epochs {EPOCHS} --lr {LR} --grad-accum 16 --max-len 1024 --out models/full

In [ ]:
!python train/predict.py --adapter models/full/adapter --split val --gold-only     --out data/predictions/student_full_val_gold.jsonl
!python label/gold.py score data/predictions/student_full_val_gold.jsonl

In [ ]:
for src, dst in (("models/full", "/kaggle/working/models/full"),
                 ("data/predictions", "/kaggle/working/predictions")):
    shutil.copytree(src, dst, dirs_exist_ok=True)
print(os.listdir("/kaggle/working/models/full"), os.listdir("/kaggle/working/predictions"))